In [ ]:
# source: https://github.com/ahans30/Binoculars
from Binoculars.binoculars.detector import Binoculars

In [ ]:
from tqdm import tqdm_notebook as tqdm
bino = Binoculars(
    observer_name_or_path='Qwen/Qwen2.5-7B',
    performer_name_or_path='Qwen/Qwen2.5-7B-Instruct'
)

In [ ]:
# ChatGPT (GPT-4) output when prompted with “Can you write a few sentences about a capybara that is an astrophysicist?"
sample_string = '''Dr. Capy Cosmos, a capybara unlike any other, astounded the scientific community with his
groundbreaking research in astrophysics.'''

print(bino.compute_score(sample_string))  # 0.75661373
print(bino.predict(sample_string))  # 'Most likely AI-Generated'

In [ ]:
import joblib
from tqdm import tqdm
import pandas as pd

train = pd.read_parquet("../../../data/train_data.parquet")
train_val = train[train.is_val == True]
print("Number of samples to process train:", len(train_val))

test = pd.read_parquet("../../../data/test_data.parquet")
print("Number of samples to process test:", len(test))

In [ ]:
from tqdm import tqdm
import torch
import gc

batch_size = 16

score_original, score_generated = [], []
for start in tqdm(range(0, len(train_val), batch_size), desc="Progress"):
    end = start + batch_size
    batch_real = train_val["real_text"].iloc[start:end].tolist()
    batch_generated = train_val["generated_text"].iloc[start:end].tolist()

    # Compute scores in batch
    score_original.extend(bino.compute_score(batch_real))
    score_generated.extend(bino.compute_score(batch_generated))

    # Clean cache:
    del batch_real, batch_generated
    torch.cuda.empty_cache()
    gc.collect()

    if len(score_original) % 10000 == 0:
      data_sample = train_val.iloc[:len(score_original)]
      data_sample["bino_real"] = score_original
      data_sample["bino_generated"] = score_generated
      data_sample.to_parquet("train_val_bino_scores.parquet")

train_val["bino_real"] = score_original
train_val["bino_generated"] = score_generated
train_val.to_parquet("train_val_bino_scores.parquet")

In [ ]:
score_original, score_generated = [], []
for start in tqdm(range(0, len(test), batch_size), desc="Progress"):
    end = start + batch_size
    batch_real = test["real_text"].iloc[start:end].tolist()
    batch_generated = test["generated_text"].iloc[start:end].tolist()

    # Compute scores in batch
    score_original.extend(bino.compute_score(batch_real))
    score_generated.extend(bino.compute_score(batch_generated))

    # Clean cache:
    del batch_real, batch_generated
    torch.cuda.empty_cache()
    gc.collect()

    if len(score_original) % 10000 == 0:
      data_sample = test.iloc[:len(score_original)]
      data_sample["bino_real"] = score_original
      data_sample["bino_generated"] = score_generated
      data_sample.to_parquet("test_bino_scores.parquet")

test["bino_real"] = score_original
test["bino_generated"] = score_generated
test.to_parquet("test_bino_scores.parquet")